In [8]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.base import clone, BaseEstimator, TransformerMixin
from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold
from sklearn.metrics import root_mean_squared_log_error, root_mean_squared_error
#root_mean_squared_log_error(actual, predicted)

from sklearn.linear_model import Ridge, ElasticNet, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.model_selection import StratifiedKFold

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor, LGBMClassifier
from catboost import CatBoostRegressor, CatBoostClassifier

warnings.filterwarnings("ignore")

In [13]:
df = pd.read_csv('train.csv')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.precision', 2)
pd.options.display.float_format = '{:.2f}'.format

In [14]:
df.head()

,project_id,num_commits,active_commit_days,project_span_days,commit_cadence,pct_commits_weekend,pct_commits_late_night,lines_added_total,lines_deleted_total,churn_ratio,median_lines_per_commit,max_lines_in_one_commit,pct_code_in_first_commit,num_commit_authors,avg_commit_msg_length,pct_generic_commit_msgs,num_branches,num_merge_commits,lines_of_code,num_source_files,num_directories,max_dir_depth,num_languages,primary_language,pct_primary_language,num_dependencies,framework,uses_game_engine,has_tests,num_test_files,has_ci_config,has_dockerfile,has_backend,has_database,avg_file_length,comment_line_ratio,num_image_files,num_audio_files,num_3d_model_files,num_font_files,asset_size_mb,has_custom_art,readme_word_count,readme_num_headings,readme_num_images,readme_num_code_blocks,readme_has_install_instructions,has_license,has_docs_folder,url_is_live,hosting_type,page_weight_kb,num_js_files_loaded,num_internal_pages,is_mobile_responsive,has_custom_domain,load_time_ms,description_word_count,project_category,is_multiplayer,mentions_ai_api,is_clone_or_tutorial,stars,forks,num_issues,num_pull_requests,num_releases,is_forked_from_template,author_prior_projects,author_account_age_years,hours_spent,log_hours,hours_bucket
0,535ce026f7,44,15,56,daily,0.20,0.45,8500,2900,0.34,128.50,1600,0.03,1,38.73,0.02,1,0,4600,43,23,5,5,TypeScript,0.95,20,nextjs,False,False,0,False,True,True,True,107.28,0.02,8,0,0,0,0.10,False,401,17,0,5,True,True,False,False,hack-club,NaN,NaN,NaN,NaN,False,NaN,46,app,False,False,False,2,0,0,0,0,False,0,1-2,32.20,3.47,30-100
1,869ba41ed7,9,3,76,sporadic,0.89,0.00,500,100,0.25,32.00,400,0.07,2,17.56,0.78,1,0,300,2,0,0,1,Python,1.00,0,none,False,False,0,False,False,False,False,152.00,0.11,0,3,0,0,3.00,False,65,1,0,0,True,False,False,True,itch,680.00,3.00,2.00,True,False,800.00,9,game,False,False,False,1,0,0,0,0,False,0,3-5,5.00,1.61,<10
2,d6f81f8862,12,1,0,frequent,1.00,0.00,5900,100,0.02,64.50,3100,0.12,3,24.00,0.33,1,0,1300,5,0,0,3,JavaScript,0.84,0,godot,True,False,0,False,False,False,False,269.80,0.34,13,0,0,0,2.50,True,16,1,0,0,False,False,False,True,itch,680.00,3.00,3.00,True,False,300.00,11,game,False,False,False,0,0,0,0,0,False,0,3-5,1.30,0.26,<10
3,152b5d915d,4,1,0,frequent,0.00,0.25,6000,0,0.00,1904.00,2200,0.27,1,18.50,1.00,1,0,300,20,0,0,1,GDScript,1.00,0,godot,True,False,0,False,False,False,False,14.75,0.00,85,0,0,1,0.50,True,0,0,0,0,False,True,False,True,itch,680.00,3.00,2.00,True,False,300.00,6,game,False,False,False,0,0,0,0,0,False,0,<1,2.00,0.69,<10
4,3b5f8a49e6,7,3,64,sporadic,0.86,0.00,800,0,0.04,198.00,200,0.25,1,55.86,0.43,1,0,600,3,0,0,3,JavaScript,0.49,0,none,False,False,0,False,False,False,False,216.67,0.04,0,0,0,0,0.00,False,89,1,2,0,False,True,False,False,github-pages,NaN,NaN,NaN,NaN,False,NaN,41,app,False,False,False,0,0,0,0,0,False,0,1-2,4.00,1.39,<10


In [ ]:
X = df.drop(columns=["project_id", "hours_spent", "log_hours", "hours_bucket"])
y = df["hours_spent"]

In [16]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
folds = list(skf.split(X, df["hours_bucket"])) # list of (train_idx, val_idx)